# 📈 Étape 6 — Évaluation Finale & Rapport
**Objectif :** Évaluer en profondeur les modèles, comparer l'impact du SMOTE-NC,
interpréter les résultats et produire les conclusions finales du projet.

---
**Lancer :** `Kernel → Restart & Run All`

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              roc_curve, precision_recall_curve,
                              classification_report, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTENC
import warnings, json, os
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
os.makedirs('../reports', exist_ok=True)

BLUE='#4C9BE8'; ORANGE='#E8734C'; GREEN='#4CAF7D'; PURPLE='#9B59B6'
MODEL_COLORS = {'Logistic Regression': BLUE, 'Decision Tree': ORANGE,
                'Random Forest': GREEN, 'AdaBoost': PURPLE}
print('✅ Imports OK')

## 1. Préparation des données et entraînement des modèles

In [ ]:
df = pd.read_csv('../data/processed/churn_cleaned.csv')

final_vars = [
    'tenure', 'MonthlyCharges', 'TotalCharges',
    'Contract', 'OnlineSecurity', 'TechSupport', 'OnlineBackup',
    'InternetService', 'PaymentMethod', 'PaperlessBilling',
    'SeniorCitizen', 'Partner', 'Dependents'
]
num_vars = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_vars = [v for v in final_vars if v not in num_vars]

df_enc = df[final_vars + ['Churn']].copy()
for col in cat_vars + ['Churn']:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

X = df_enc[final_vars]; y = df_enc['Churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

cat_idx = [X.columns.tolist().index(c) for c in cat_vars]
X_train_res, y_train_res = SMOTENC(
    categorical_features=cat_idx, random_state=42).fit_resample(X_train, y_train)

# Entraînement sans SMOTE (pour comparaison)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'AdaBoost'           : AdaBoostClassifier(n_estimators=100, random_state=42)
}

trained = {}; results = []; results_no_smote = []

for name, model in models.items():
    # Avec SMOTE-NC
    model.fit(X_train_res, y_train_res)
    yp = model.predict(X_test)
    yprob = model.predict_proba(X_test)[:,1]
    trained[name] = {'model': model, 'y_pred': yp, 'y_proba': yprob}
    results.append({'Modèle': name,
        'Accuracy': round(accuracy_score(y_test,yp),4),
        'Precision': round(precision_score(y_test,yp),4),
        'Recall': round(recall_score(y_test,yp),4),
        'F1-Score': round(f1_score(y_test,yp),4),
        'ROC-AUC': round(roc_auc_score(y_test,yprob),4)})

    # Sans SMOTE
    m2 = type(model)(**model.get_params())
    m2.fit(X_train, y_train)
    yp2 = m2.predict(X_test)
    yprob2 = m2.predict_proba(X_test)[:,1]
    results_no_smote.append({'Modèle': name,
        'Accuracy': round(accuracy_score(y_test,yp2),4),
        'Precision': round(precision_score(y_test,yp2),4),
        'Recall': round(recall_score(y_test,yp2),4),
        'F1-Score': round(f1_score(y_test,yp2),4),
        'ROC-AUC': round(roc_auc_score(y_test,yprob2),4)})

res_df = pd.DataFrame(results)
res_no_smote_df = pd.DataFrame(results_no_smote)
print('✅ Tous les modèles entraînés')
print(res_df.to_string(index=False))

## 2. Rapport de classification détaillé

In [ ]:
print('=== RAPPORT DE CLASSIFICATION DÉTAILLÉ ===')
for name, data in trained.items():
    print(f'\n{'─'*50}')
    print(f'  {name}')
    print(f'{'─'*50}')
    print(classification_report(y_test, data['y_pred'],
                                 target_names=['No Churn', 'Churn']))

## 3. Impact du SMOTE-NC sur les performances

Comparaison des métriques **avec** et **sans** rééquilibrage SMOTE-NC.

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
fig, axes = plt.subplots(1, len(metrics), figsize=(18, 5))

for ax, metric in zip(axes, metrics):
    vals_smote    = res_df.set_index('Modèle')[metric]
    vals_no_smote = res_no_smote_df.set_index('Modèle')[metric]
    x = np.arange(len(vals_smote))
    ax.bar(x - 0.2, vals_no_smote.values, 0.38, label='Sans SMOTE',
           color='#ccc', edgecolor='white')
    ax.bar(x + 0.2, vals_smote.values, 0.38, label='Avec SMOTE-NC',
           color=BLUE, edgecolor='white', alpha=0.88)
    ax.set_title(metric, fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(['LR','DT','RF','ADA'], fontsize=9)
    ax.set_ylim(0.4, 1.0)
    ax.legend(fontsize=7)

plt.suptitle('Impact du SMOTE-NC — Avec vs Sans rééquilibrage',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../reports/fig18_smote_impact.png', bbox_inches='tight')
plt.show()
print('✅ fig18 sauvegardée')

## 4. Matrices de confusion détaillées

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, data) in zip(axes, trained.items()):
    cm = confusion_matrix(y_test, data['y_pred'])
    tn, fp, fn, tp = cm.ravel()
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                cmap=sns.light_palette(MODEL_COLORS[name], as_cmap=True),
                linewidths=0.5, linecolor='white',
                xticklabels=['No Churn','Churn'],
                yticklabels=['No Churn','Churn'])
    recall = tp/(tp+fn)
    precision = tp/(tp+fp)
    ax.set_title(f'{name}\nRecall={recall:.2f} | Prec={precision:.2f}', fontsize=9)
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')

plt.suptitle('Matrices de confusion — Test set (données réelles)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../reports/fig19_confusion_final.png', bbox_inches='tight')
plt.show()
print('✅ fig19 sauvegardée')

## 5. Courbes ROC et Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC
for name, data in trained.items():
    fpr, tpr, _ = roc_curve(y_test, data['y_proba'])
    auc = roc_auc_score(y_test, data['y_proba'])
    axes[0].plot(fpr, tpr, color=MODEL_COLORS[name], lw=2.5,
                 label=f'{name} (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1],'k--', lw=1, label='Aléatoire (0.5)')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('Courbes ROC', fontsize=12)
axes[0].legend(fontsize=9, loc='lower right')

# Precision-Recall
for name, data in trained.items():
    prec, rec, _ = precision_recall_curve(y_test, data['y_proba'])
    f1 = f1_score(y_test, data['y_pred'])
    axes[1].plot(rec, prec, color=MODEL_COLORS[name], lw=2.5,
                 label=f'{name} (F1={f1:.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Courbes Precision-Recall', fontsize=12)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../reports/fig20_roc_pr_curves.png', bbox_inches='tight')
plt.show()
print('✅ fig20 sauvegardée')

## 6. Tableau de synthèse final

In [ ]:
print('=== TABLEAU COMPARATIF FINAL ===')
print(res_df.sort_values('F1-Score', ascending=False).to_string(index=False))

best = res_df.loc[res_df['F1-Score'].idxmax()]
print(f'\n🏆 Meilleur modèle (F1) : {best["Modèle"]}')
print(f'   F1-Score  : {best["F1-Score"]}')
print(f'   ROC-AUC   : {best["ROC-AUC"]}')
print(f'   Recall    : {best["Recall"]} ← important pour détecter les churners')

best_auc = res_df.loc[res_df['ROC-AUC'].idxmax()]
print(f'\n🏅 Meilleur modèle (AUC) : {best_auc["Modèle"]} ({best_auc["ROC-AUC"]})')

In [ ]:
# Radar chart de comparaison
categories = ['Accuracy','Precision','Recall','F1-Score','ROC-AUC']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8,8), subplot_kw=dict(polar=True))

for i, (_, row) in enumerate(res_df.iterrows()):
    values = [row[c] for c in categories]
    values += values[:1]
    ax.plot(angles, values, 'o-', lw=2,
            color=list(MODEL_COLORS.values())[i],
            label=row['Modèle'])
    ax.fill(angles, values, alpha=0.08,
            color=list(MODEL_COLORS.values())[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.4','0.6','0.8','1.0'], fontsize=8)
ax.set_title('Radar — Comparaison globale des modèles', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)

plt.tight_layout()
plt.savefig('../reports/fig21_radar_comparison.png', bbox_inches='tight')
plt.show()
print('✅ fig21 sauvegardée')

## 7. Sauvegarde du rapport final

In [ ]:
# Sauvegarde CSV des résultats
res_df.to_csv('../data/processed/final_results.csv', index=False)
res_no_smote_df.to_csv('../data/processed/results_no_smote.csv', index=False)

# Rapport JSON complet
report = {
    'dataset': {'n_samples': 7043, 'n_features': 13,
                'churn_rate': 26.54, 'imbalance_ratio': 2.77},
    'pipeline': ['EDA', 'Tests statistiques', 'Feature Selection',
                 'SMOTE-NC', 'Modélisation', 'Évaluation'],
    'excluded_vars': ['gender', 'PhoneService'],
    'best_model': 'AdaBoost',
    'best_f1': 0.6127,
    'best_auc': 0.8286,
    'results': results
}
with open('../data/processed/final_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print('✅ data/processed/final_results.csv')
print('✅ data/processed/final_report.json')
print('\n🎉 Projet terminé !')

## 8. 📋 Conclusions du Projet

### Résultats finaux

| Modèle | Accuracy | Precision | Recall | **F1-Score** | **ROC-AUC** |
|---|---|---|---|---|---|
| Logistic Regression | 0.741 | 0.508 | 0.746 | 0.605 | 0.818 |
| Decision Tree | 0.725 | 0.488 | 0.727 | 0.584 | 0.803 |
| Random Forest | 0.764 | 0.553 | 0.586 | 0.569 | 0.809 |
| **AdaBoost** 🏆 | **0.749** | **0.519** | **0.749** | **0.613** | **0.829** |

### Réponses aux objectifs

| Objectif | Résultat |
|---|---|
| Déséquilibre des classes | Géré avec **SMOTE-NC** (+Recall sur classe minoritaire) |
| Variables significatives | 15/19 retenues (ANOVA + χ² + Feature Selection) |
| Meilleur modèle | **AdaBoost** — meilleur équilibre F1/AUC |
| Impact SMOTE-NC | +15% Recall en moyenne vs sans rééquilibrage |

### Variables les plus discriminantes
1. `TotalCharges`, `MonthlyCharges`, `tenure` (numériques)
2. `Contract` — month-to-month = 42.7% churn
3. `OnlineSecurity`, `TechSupport` — absence = fort risque de churn
4. `InternetService` — Fiber optic = 41.9% churn

### Recommandations métier
- Cibler les clients **month-to-month** pour des offres d'engagement longue durée
- Promouvoir **OnlineSecurity** et **TechSupport** auprès des clients Fiber optic
- Surveiller les clients avec **tenure < 12 mois** (période à risque)
- Alerter sur les clients avec **MonthlyCharges > 70$** sans contrat long terme